# TP 4 — Atelier décorateurs

In [1]:
from functools import wraps, lru_cache
from time import perf_counter, sleep

# Q1
def chronometrer(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        debut = perf_counter()
        resultat = fonction(*args, **kwargs)
        print(f"{fonction.__name__} a pris {perf_counter() - debut:.2f} s")
        return resultat
    return enveloppe

@chronometrer
def lente():
    sleep(1)
    return "Terminé"

print(lente())

# Q2
def majuscules(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        resultat = fonction(*args, **kwargs)
        return resultat.upper() if isinstance(resultat, str) else resultat
    return enveloppe

@majuscules
def bonjour():
    return "bonjour"

print(bonjour())

# Q3
def compter_appels(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        enveloppe.appels += 1
        return fonction(*args, **kwargs)
    enveloppe.appels = 0
    return enveloppe

@compter_appels
def addition(a, b):
    return a + b

addition(1, 2); addition(3, 4); addition(5, 6)
print("Nombre d'appels :", addition.appels)

# Q4
def verifier_positifs(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        for valeur in list(args) + list(kwargs.values()):
            if isinstance(valeur, (int, float)) and valeur < 0:
                raise ValueError("Les arguments numériques doivent être positifs ou nuls.")
        return fonction(*args, **kwargs)
    return enveloppe

@verifier_positifs
def surface(longueur, largeur):
    return longueur * largeur

print("Surface :", surface(5, 3))
try:
    surface(-5, 3)
except ValueError as erreur:
    print("Erreur attendue :", erreur)

# Q5 — Décorateur avec paramètre
def chronometrer_unite(unite="s"):
    if unite not in ("s", "ms"):
        raise ValueError("Utiliser 's' ou 'ms'.")
    def decorateur(fonction):
        @wraps(fonction)
        def enveloppe(*args, **kwargs):
            debut = perf_counter()
            resultat = fonction(*args, **kwargs)
            duree = perf_counter() - debut
            duree = duree * 1000 if unite == "ms" else duree
            print(f"{fonction.__name__} : {duree:.2f} {unite}")
            return resultat
        return enveloppe
    return decorateur

@chronometrer_unite(unite="ms")
def calcul():
    return sum(range(10000))
calcul()

lente a pris 1.00 s
Terminé
BONJOUR
Nombre d'appels : 3
Surface : 15
Erreur attendue : Les arguments numériques doivent être positifs ou nuls.
calcul : 0.27 ms


49995000

In [2]:
# Q6 — Cache manuel ; les arguments doivent être hachables
def memoriser(fonction):
    cache = {}
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        cle = (args, tuple(sorted(kwargs.items())))
        if cle not in cache:
            cache[cle] = fonction(*args, **kwargs)
        return cache[cle]
    enveloppe.cache = cache
    return enveloppe

@memoriser
def fibonacci_memo(n):
    if n < 2:
        return n
    return fibonacci_memo(n - 1) + fibonacci_memo(n - 2)

@lru_cache(maxsize=None)
def fibonacci_lru(n):
    if n < 2:
        return n
    return fibonacci_lru(n - 1) + fibonacci_lru(n - 2)

def fibonacci_simple(n):
    if n < 2:
        return n
    return fibonacci_simple(n - 1) + fibonacci_simple(n - 2)

for nom, fonction in [
    ("sans cache", fibonacci_simple),
    ("cache manuel", fibonacci_memo),
    ("lru_cache", fibonacci_lru),
]:
    debut = perf_counter()
    resultat = fonction(30)
    print(nom, ":", resultat, f"en {perf_counter() - debut:.6f} s")

# Q7 — Les deux ordres d'empilement
@chronometrer
@compter_appels
def ordre_a(n):
    return sum(range(n))

@compter_appels
@chronometrer
def ordre_b(n):
    return sum(range(n))

ordre_a(10000); ordre_a(10000)
ordre_b(10000); ordre_b(10000)
print("Compteur ordre A :", ordre_a.appels)
print("Compteur ordre B :", ordre_b.appels)

sans cache : 832040 en 0.145798 s
cache manuel : 832040 en 0.000051 s
lru_cache : 832040 en 0.000013 s
ordre_a a pris 0.00 s
ordre_a a pris 0.00 s
ordre_b a pris 0.00 s
ordre_b a pris 0.00 s
Compteur ordre A : 0
Compteur ordre B : 2


In [3]:
# Q8 — MiniFlask
class MiniFlask:
    def __init__(self):
        self.routes = {}
        self.vue_404 = lambda: "404 : page introuvable"

    def route(self, chemin):
        def enregistrer(fonction):
            if chemin in self.routes:
                raise ValueError(f"Route déjà enregistrée : {chemin}")
            self.routes[chemin] = fonction
            return fonction
        return enregistrer

    def erreur_404(self, fonction):
        self.vue_404 = fonction
        return fonction

    def lister_routes(self):
        for chemin in self.routes:
            print(chemin)

    def traiter(self, chemin):
        fonction = self.routes.get(chemin)
        return fonction() if fonction else self.vue_404()

app = MiniFlask()

@app.erreur_404
def page_404():
    return "404 personnalisé : page introuvable."

@app.route("/")
def accueil():
    return "Bonjour !"

@app.route("/contact")
def contact():
    return "Page contact"

print(app.traiter("/"))
print(app.traiter("/contact"))
print(app.traiter("/inconnu"))
print("Routes :")
app.lister_routes()

try:
    @app.route("/")
    def autre_accueil():
        return "Autre"
except ValueError as erreur:
    print("Doublon refusé :", erreur)

Bonjour !
Page contact
404 personnalisé : page introuvable.
Routes :
/
/contact
Doublon refusé : Route déjà enregistrée : /
